# <a id='toc1_'></a>[Analysis - renal function - infants (before year 1)](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [Analysis - renal function - infants (before year 1)](#toc1_)    
  - [Import dependencies and load data](#toc1_1_)    
  - [Descriptive statistics](#toc1_2_)    
      - [Descriptive statistics data prep](#toc1_2_1_1_)    
      - [Continuous vars descriptive statistics table](#toc1_2_1_2_)    
      - [Categorical vars descriptive statistics table](#toc1_2_1_3_)    
  - [Data exploration and review](#toc1_3_)    
      - [Overview - baseline creatinine](#toc1_3_1_1_)    
      - [Overview - VUR, Dysplasia, recurrent UTIs before the age of 1](#toc1_3_1_2_)    
    - [Distributions - categorical variables](#toc1_3_2_)    
    - [Distributions - continuous variables](#toc1_3_3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [ ]:
# logging setup
from puv.logging_setup import setup
setup()  # or DEBUG if you want more detail

import logging
logger = logging.getLogger(__name__)
logger.info("Logging initialized.")

In [ ]:
# Environment setup
from pathlib import Path

BASE = Path('/workspaces/CODESPACE/').resolve()

# Define paths for data and analysis directories & ensure key folders exist
PREP = BASE / "data" / "prep"
EDA = BASE / "results" / "eda"

for p in [PREP, EDA]:
    p.mkdir(parents=True, exist_ok=True)

logger.info("Project base set to: %s", BASE)


## <a id='toc1_1_'></a>[Import dependencies and load data](#toc0_)

In [ ]:
# import dependencies
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# load data
df_all = pd.read_excel(PREP / 'data_all.xlsx')
df_processed = pd.read_excel(PREP / 'data_processed.xlsx')
df_renal = pd.read_excel(PREP / 'renal_set_infants.xlsx')

# Data check
# display(sorted(df_renal.columns.to_list()))

plt.rcParams.update({
    "font.size": 12, "axes.titlesize": 15, "axes.labelsize": 13,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 12,
})


## <a id='toc1_2_'></a>[Descriptive statistics](#toc0_)

#### <a id='toc1_2_1_1_'></a>[Descriptive statistics data prep](#toc0_)

In [ ]:
# replace the z values with raw values
df_dstats_raw = df_all[[    "Patient number",
                            "1 Weight (1 year, kg)"]].copy()

# Select all further columns that do not start with 'Z ' using regex, as we'll use the raw data for descriptive stats
df_dstats_regex = df_renal.filter(regex='^(?!Z ).*').copy()

# Merge the two dataframes on 'Patient number'
df_dstats = pd.merge(df_dstats_raw, df_dstats_regex, on='Patient number', how='inner')

# Drop the Patient number column as it's not needed for descriptive stats
df_dstats.drop(columns=['Patient number'], inplace=True)

# Separate the df based on categorical and continuous variables
df_dstats_cats = df_dstats.loc[:, df_dstats.nunique(dropna=True) <=2].copy()
df_dstats_conts = df_dstats.loc[:, df_dstats.nunique(dropna=True) >2].copy()

# Convert all columns in continuous dataframe to numeric, coercing errors to NaN
df_dstats_conts = df_dstats_conts.apply(pd.to_numeric, errors="coerce")

# Convert each continuous explicitly to a numeric column, keeping NaN
for col in df_dstats_conts.columns:
    # convert the column to 'float64' to handle NaN values properly
    df_dstats_conts[col] = df_dstats_conts[col].astype('float64')
    
# Convert each categorical explicitly to a numeric column, keeping NaN
for col in df_dstats_cats.columns:
    # convert the column to 'Int64' to handle NaN values properly
    df_dstats_cats[col] = df_dstats_cats[col].astype('Int64')

df_dstats_conts.drop(columns=['Weight (1 year)'], inplace=True)
df_dstats_conts.rename(columns={"1 Weight (1 year, kg)": "Weight (1 year)"}, inplace=True)


#### <a id='toc1_2_1_2_'></a>[Continuous vars descriptive statistics table](#toc0_)

In [ ]:
# Total number of rows in the DataFrame
total_rows = len(df_dstats_conts)

# Calculate 25th and 75th percentiles
quantile_25 = df_dstats_conts.quantile(0.25).round(2)
quantile_75 = df_dstats_conts.quantile(0.75).round(2)

# Combine percentiles in a string format
range_str = quantile_25.astype(str) + " - " + quantile_75.astype(str)

# Calculate missingness  as a percentage
missingness_percentage = (100 - (df_dstats_conts.count() / total_rows) * 100).round(1)

# Create the summary table
summary_table_conts = pd.DataFrame({
    'Mean': df_dstats_conts.mean().round(2),
    'Range (25th - 75th)': range_str,
    'Missingness': missingness_percentage.astype(str) + '%'
})

summary_table_conts.drop(index=['Weight (1 year)'], inplace=True, errors='ignore')

summary_table_conts.to_excel(EDA/ 'Infant_continuous_vars_summary_statistics.xlsx')

# view summary_table_conts 
display(summary_table_conts)


#### <a id='toc1_2_1_3_'></a>[Categorical vars descriptive statistics table](#toc0_)

In [ ]:
# Total number of rows in the DataFrame
total_rows = len(df_dstats_cats)

# Calculate missingness  as a percentage
missingness_percentage = (100 - (df_dstats_cats.count() / total_rows) * 100).round(1)

# Calculate distribution for all columns
distribution = df_dstats_cats.apply(lambda col: ' -- '.join(
    [f"{val}: {(col.value_counts(normalize=True)[val] * 100):.1f}%" for val in sorted(col.dropna().unique())]
))

# Combine results into a summary DataFrame
summary_table_cats = pd.DataFrame({
    'Distribution': distribution,
    'Missingness': missingness_percentage.astype(str) + '%'
})

summary_table_cats.to_excel(EDA / 'Infant_categorical_vars_summary_statistics.xlsx')

# view summary_table_cats 
summary_table_cats

## <a id='toc1_3_'></a>[Data exploration and review](#toc0_)

#### <a id='toc1_3_1_1_'></a>[Overview - Nadir creatinine](#toc0_)

In [ ]:
# Select the nadir creatinine and CKD features
df_nadir_creatinine =  df_all[['Nadir creatinine', 'Renal function at last follow up', 'CKD5', 'Patient number']]
df_nadir_creatinine_dropna = df_nadir_creatinine.dropna().copy()   

# Note that there are 5 patients without a Nadir creatinine value. 
# 4 of these have Renal function at last follow up = 0, and the other has Renal function at last follow up = 1.

# Create a new column 'CKD stages' based on the conditions
df_nadir_creatinine_dropna.loc[df_nadir_creatinine_dropna['CKD5'] == 1, 'CKD stages'] = '[ 5 ]'
df_nadir_creatinine_dropna.loc[(df_nadir_creatinine_dropna['CKD5'] != 1) & (df_nadir_creatinine_dropna['Renal function at last follow up'] == 1), 'CKD stages'] = '[ 3 - 4 ]'
df_nadir_creatinine_dropna.loc[(df_nadir_creatinine_dropna['CKD5'] != 1) & (df_nadir_creatinine_dropna['Renal function at last follow up'] != 1), 'CKD stages'] = '[ 1 - 2 ]'

# Select the required columns
df_nadir_crea_ckd_category = df_nadir_creatinine_dropna[['Patient number', 'Nadir creatinine', 'CKD stages']] 

# Count the unique values and their frequencies in the 'CKD stages' column
nc_ckd_stage_counts = df_nadir_crea_ckd_category['CKD stages'].value_counts()

# Display the number of unique CKD stages and their frequencies


logger.info(f"Number of unique CKD stages: {nc_ckd_stage_counts.size}")
print("Frequencies of each CKD stage:")
print(nc_ckd_stage_counts)



In [ ]:
fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(16, 10),
    sharex=True,
    gridspec_kw={"hspace": 0.2}
)

# Define common bins
n_crea_range = df_nadir_crea_ckd_category['Nadir creatinine']
bins = np.linspace(n_crea_range.min(), n_crea_range.max(), 100)

# Color map
color_map = {
    '[ 1 - 2 ]': '#ADD8E6',   # Light blue
    '[ 3 - 4 ]': '#FFB6C1',   # Light red / pink
    '[ 5 ]': '#FF0000'        # Red
}

# Explicit order (low → high CKD)
category_order = ['[ 1 - 2 ]', '[ 3 - 4 ]', '[ 5 ]']

# Plot each CKD stage in its own horizontal panel
for ax, label in zip(axes, category_order):
    subset = df_nadir_crea_ckd_category[
        df_nadir_crea_ckd_category['CKD stages'] == label
    ]

    ax.hist(
        subset['Nadir creatinine'],
        bins=bins,
        edgecolor='black',
        alpha=0.7,
        color=color_map[label]
    )

    ax.set_ylabel("Frequency", fontsize=15)
    ax.set_title(f"CKD stages: {label}", fontsize=18, loc="right")

    # CKD cutoffs
    ax.axvline(x=31, color='gray', linestyle='--', linewidth=3)
    ax.axvline(x=75, color='gray', linestyle='--', linewidth=3)

# Shared x-axis
axes[-1].set_xlabel('Nadir creatinine [μmol/L]', fontsize=18)
axes[-1].set_xticks(np.arange(0, 310, step=10))

# Global title
fig.suptitle(
    'Distribution of Nadir creatinine levels by CKD stages',
    fontsize=22,
    y=0.97
)

plt.savefig(
    EDA / 'Infant_Nadir_creatinine_vs_CKD_stages_split.png',
    format='png',
    bbox_inches='tight'
)
plt.show()

#### Overview - Baseline creatinine


In [ ]:
# Select the baseline creatinine and CKD features
df_baseline_creatinine =  df_all[['Baseline creatinine', 'Renal function at last follow up', 'CKD5', 'Patient number']]
df_baseline_creatinine_dropna = df_baseline_creatinine.dropna().copy()   

# Note that there are 5 patients without a Base creatinine value. 
# 4 of these have Renal function at last follow up = 0, and the other has Renal function at last follow up = 1.

# Create a new column 'CKD stages' based on the conditions
df_baseline_creatinine_dropna.loc[df_baseline_creatinine_dropna['CKD5'] == 1, 'CKD stages'] = '[ 5 ]'
df_baseline_creatinine_dropna.loc[(df_baseline_creatinine_dropna['CKD5'] != 1) & (df_baseline_creatinine_dropna['Renal function at last follow up'] == 1), 'CKD stages'] = '[ 3 - 4 ]'
df_baseline_creatinine_dropna.loc[(df_baseline_creatinine_dropna['CKD5'] != 1) & (df_baseline_creatinine_dropna['Renal function at last follow up'] != 1), 'CKD stages'] = '[ 1 - 2 ]'

# Select the required columns 
df_baseline_crea_ckd_category = df_baseline_creatinine_dropna[['Patient number', 'Baseline creatinine', 'CKD stages']] 

# Count the unique values and their frequencies in the 'CKD stages' column
nc_bc_ckd_stage_counts = df_baseline_crea_ckd_category['CKD stages'].value_counts()

# Display the number of unique CKD stages and their frequencies


logger.info(f"Number of unique CKD stages: {nc_bc_ckd_stage_counts.size}")
print("Frequencies of each CKD stage:")
print(nc_bc_ckd_stage_counts)

In [ ]:
fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(16, 10),
    sharex=True,
    gridspec_kw={"hspace": 0.25}
)

# Common bins
n_crea_range = df_baseline_crea_ckd_category['Baseline creatinine']
bins = np.linspace(n_crea_range.min(), n_crea_range.max(), 50)

# Color map
color_map = {
    '[ 1 - 2 ]': '#ADD8E6',   # Light blue
    '[ 3 - 4 ]': '#FFB6C1',   # Light red / pink
    '[ 5 ]': '#FF0000'        # Red
}

# Explicit CKD order
category_order = ['[ 1 - 2 ]', '[ 3 - 4 ]', '[ 5 ]']

# Plot each CKD stage in its own panel
for ax, label in zip(axes, category_order):
    subset = df_baseline_crea_ckd_category[
        df_baseline_crea_ckd_category['CKD stages'] == label
    ]

    ax.hist(
        subset['Baseline creatinine'],
        bins=bins,
        edgecolor='black',
        alpha=0.7,
        color=color_map[label]
    )

    ax.set_ylabel("Frequency", fontsize=15)
    ax.set_title(f"CKD stages: {label}", fontsize=18, loc="left")

# Shared x-axis
axes[-1].set_xlabel('Baseline creatinine [μmol/L]', fontsize=18)
axes[-1].set_xticks(np.arange(0, 510, step=20))

# Global title
fig.suptitle(
    'Distribution of Baseline creatinine levels by CKD stages',
    fontsize=22,
    y=0.97
)

plt.savefig(
    EDA / 'Infant_Baseline_creatinine_vs_CKD_stages_split.png',
    format='png',
    bbox_inches='tight'
)
plt.show()

#### <a id='toc1_3_1_2_'></a>[Overview - VUR, Dysplasia, recurrent UTIs before the age of 1](#toc0_)

In [ ]:
# Select the required columns for VUR, Dysplasia and UTI exploration
df_vur_dusplasia_uti = df_processed [[
        'Patient number',
        'CKD5',
        'Renal function at last follow up',
        'No VUR',
        'Unilateral VUR',
        'Bilateral VUR',
        'No dysplasia',
        'Unilateral dysplasia',
        'Bilateral dysplasia',
        'Rec. UTI < 1 year'
    ]].copy(deep=True)

# Create a new column 'CKD stages' based on the conditions
df_vur_dusplasia_uti.loc[df_vur_dusplasia_uti['CKD5'] == 1, 'CKD stages'] = '[ 5 ]'
df_vur_dusplasia_uti.loc[(df_vur_dusplasia_uti['CKD5'] != 1) & (df_vur_dusplasia_uti['Renal function at last follow up'] == 1), 'CKD stages'] = '[ 3 - 4 ]'
df_vur_dusplasia_uti.loc[(df_vur_dusplasia_uti['CKD5'] != 1) & (df_vur_dusplasia_uti['Renal function at last follow up'] != 1), 'CKD stages'] = '[ 1 - 2 ]'

# Drop CKD5 and renal function at last follow up columns
df_vur_dusplasia_uti.drop(columns=['CKD5', 'Renal function at last follow up', 'Patient number'], inplace=True)


# Display the new table to check
# display (df_vur_dusplasia_uti)

In [ ]:
# Visualize the pairwise relationships between the variables
# Extract variable names excluding the label
label_var = 'CKD stages'
plot_vars = [col for col in df_vur_dusplasia_uti.columns if col != label_var]

# Define a color map for the labels
color_map = {
    '[ 5 ]': '#FF0000',  # Red
    '[ 3 - 4 ]': '#FFB6C1',  # Light red/pink
    '[ 1 - 2 ]': '#ADD8E6'  # Light blue
}

category_order = sorted(df_vur_dusplasia_uti[label_var].unique())

# Number of variables
n_vars = len(plot_vars)

# Set up the figure for subplots
fig, axes = plt.subplots(n_vars, n_vars, figsize=(25, 20))

# Iterate over all pairs of variables
for i, var1 in enumerate(plot_vars):
    for j, var2 in enumerate(plot_vars):
        ax = axes[i, j]
        
        if i == j:
            # For diagonal plots, display the variable name
            ax.text(0.5, 0.5, var1, fontsize=18, ha='center', va='center', transform=ax.transAxes)
            ax.set_xticks([0,1])
            ax.set_yticks([0,1])
        else:
            # Create scatter plots for off-diagonal cells
            for label in category_order:
                subset = df_vur_dusplasia_uti[df_vur_dusplasia_uti[label_var] == label]
                
                # Adding jitter by adding a small random noise to x and y values
                jitter_x = subset[var1] + np.random.normal(0, 0.08, size=len(subset))
                jitter_y = subset[var2] + np.random.normal(0, 0.08, size=len(subset))
                
                # Add scatter plot without labeling for legend
                ax.scatter(
                    jitter_x, jitter_y, alpha=0.6, 
                    color=color_map[label]
                )

            # Set axis ticks to 0 and 1 for all plots
            ax.set_xticks([0,1])
            ax.set_yticks([0,1])
            
            if i == n_vars - 1:
                ax.set_xlabel(var2, fontsize=15)
            else:
                ax.set_xticks([0,1])
            
            if j == 0:
                ax.set_ylabel(var1, fontsize=15)
            else:
                ax.set_yticks([0,1])

# Adjust layout
fig.tight_layout()

plt.legend(
    [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map[label], markersize=10, label=label) for label in category_order],
    category_order,
    bbox_to_anchor=(-6.2, 8.8),
    fontsize=18,
    title="CKD stages",
    title_fontsize=20,  # Increase title font size
)

# Add the title and subtitle
plt.suptitle("Pairwise Scatter Plots of VUR, Dysplasia, and UTI variables", fontsize=45, y=1.05)
plt.savefig(EDA / 'Infant_Pairwise_scatter_plots_VUR_Dysplasia_UTI.png', format='png', bbox_inches='tight')
# Show the plot
plt.show()


### <a id='toc1_3_2_'></a>[Distributions - categorical variables](#toc0_)

In [ ]:
# Select continuous variables
categorical_cols = df_dstats_cats.columns.to_list()

# Make sure these always appear in the CKD stage definition
required_cols = ['CKD5', 'Renal function at last follow up', 'Patient number']

# Remove 'CKD5' and 'Renal function at last follow up' from plotting list
plot_cols = [col for col in categorical_cols if col not in required_cols]

# Loop to assign CKD stages for the whole dataframe
df_ckd = df_processed[required_cols + plot_cols].dropna(subset=['CKD5', 'Renal function at last follow up']).copy()

df_ckd.loc[df_ckd['CKD5'] == 1, 'CKD stages'] = '[ 5 ]'
df_ckd.loc[(df_ckd['CKD5'] != 1) & (df_ckd['Renal function at last follow up'] == 1), 'CKD stages'] = '[ 3 - 4 ]'
df_ckd.loc[(df_ckd['CKD5'] != 1) & (df_ckd['Renal function at last follow up'] != 1), 'CKD stages'] = '[ 1 - 2 ]'

# Setup the plot grid
num_vars = len(plot_cols)
grid_size = int(np.ceil(np.sqrt(num_vars)))

fig, axes = plt.subplots(grid_size, grid_size, figsize=(grid_size * 5, grid_size * 4))
fig.subplots_adjust(hspace=0.4, wspace=0.3)  # Increase vertical and horizontal padding


category_order = sorted(df_ckd['CKD stages'].unique())
axes = axes.flatten()

# Plot bar plots for each binary variable
for idx, col in enumerate(plot_cols):
    ax = axes[idx]
    
    # Calculate the count of 0s and 1s per CKD stage
    counts = df_ckd.groupby('CKD stages')[col].value_counts().unstack(fill_value=0).reindex(category_order)
    
    # Width and position adjustment
    x = np.arange(len(counts.index))
    width = 0.35
    
    ax.bar(x - width/2, counts[0], width, label='0', color='#D3D3D3', edgecolor='black')
    ax.bar(x + width/2, counts[1], width, label='1', color='#000000', edgecolor='black')
    
    ax.set_title(col, fontsize=15)
    ax.set_xlabel('CKD Stages')
    ax.set_ylabel('Count')
    ax.set_xticks(x)
    ax.set_xticklabels(counts.index)

# Hide empty subplots
for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

# Add a single legend for the entire figure
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', ncol=2, frameon=False, fontsize=12)

# Save the figure
output_path = EDA / 'Infant_All_Binary_Variables_vs_CKD_stages.png'
fig.suptitle('Distribution of Categorical Variables by CKD Stages', fontsize=20, y=0.95)
fig.savefig(output_path, format='png', bbox_inches='tight')

plt.show()



### <a id='toc1_3_3_'></a>[Distributions - continuous variables](#toc0_)

In [ ]:
# Select continuous variables
continuous_cols = df_dstats_conts.columns.to_list()

# Make sure these always appear in the CKD stage definition
required_cols = ['CKD5', 'Renal function at last follow up', 'Patient number']

# Remove 'CKD5' and 'Renal function at last follow up' from plotting list
plot_cols = [col for col in continuous_cols if col not in required_cols]

df_all_rn = df_all.rename(columns={'1 Weight (1 year, kg)': 'Weight (1 year)'})

# Loop to assign CKD stages for the whole dataframe
df_ckd = df_all_rn[required_cols + plot_cols].dropna(subset=['CKD5', 'Renal function at last follow up']).copy()


df_ckd.loc[df_ckd['CKD5'] == 1, 'CKD stages'] = '[ 5 ]'
df_ckd.loc[(df_ckd['CKD5'] != 1) & (df_ckd['Renal function at last follow up'] == 1), 'CKD stages'] = '[ 3 - 4 ]'
df_ckd.loc[(df_ckd['CKD5'] != 1) & (df_ckd['Renal function at last follow up'] != 1), 'CKD stages'] = '[ 1 - 2 ]'

# Setup the plot grid
num_vars = len(plot_cols)
grid_size = int(np.ceil(np.sqrt(num_vars)))

fig, axes = plt.subplots(grid_size, grid_size, figsize=(grid_size * 5, grid_size * 4))
fig.subplots_adjust(hspace=0.4, wspace=0.3)  # Increase vertical and horizontal padding


# Color map for CKD stages
color_map = {
    '[ 5 ]': '#FF0000',
    '[ 3 - 4 ]': '#FFB6C1',
    '[ 1 - 2 ]': '#ADD8E6'
}

category_order = sorted(df_ckd['CKD stages'].unique())
axes = axes.flatten()

# Plot histograms for each continuous variable
for idx, col in enumerate(plot_cols):
    ax = axes[idx]
    data_range = df_ckd[col].dropna()
    bins = np.linspace(data_range.min(), data_range.max(), 50)  # Fewer bins for clarity
    
    for label in category_order:
        subset = df_ckd[df_ckd['CKD stages'] == label]
        ax.hist(subset[col], bins=bins, alpha=0.5, label=f'Stage {label}', edgecolor='black', color=color_map[label])
    
    ax.set_title(col, fontsize=15)
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    
# Hide empty subplots
for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

# Add a single legend for all subplots
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', ncol=3, frameon=False, fontsize=10 )


# Save the figure
output_path = EDA / 'Infant_All_Continuous_Variables_vs_CKD_stages.png'
fig.suptitle('Distribution of Continuous Variables by CKD Stages', fontsize=18,  y=0.95)
fig.savefig(output_path, format='png', bbox_inches='tight')

plt.show()
